In [0]:
from pyspark.sql import SparkSession
import re
import os
import boto3
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import lit, col, when

# Initialize Spark session
spark = SparkSession.builder.appName("CompareGZandCTL").getOrCreate()
s3 = boto3.client('s3')

weekly_base_path = "s3a://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly/"
ctl_keywords = ["ODM.EDW.VEN.REFERENCE.WEEKLY"]
latest_processed_path = "/dbfs/tmp/latest_processed.json"
incomplete_ctl_path = "/dbfs/tmp/incomplete_ctl.json"

# Load state files
latest_processed = {}
if os.path.exists(latest_processed_path):
    with open(latest_processed_path, 'r') as f:
        latest_processed = json.load(f)

incomplete_ctl = {}
if os.path.exists(incomplete_ctl_path):
    with open(incomplete_ctl_path, 'r') as f:
        incomplete_ctl = json.load(f)

start_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
now = datetime.now()

# Get latest dt folder
folders = [f.path for f in dbutils.fs.ls(weekly_base_path) if "dt=" in f.name]
if not folders:
    print("No folders to process.")
    exit(1)

latest_folder = sorted(folders, reverse=True)[0]
print(f"Latest folder: {latest_folder}")

files = dbutils.fs.ls(latest_folder)
ctl_files = [f for f in files if f.name.endswith(".ctl") and any(k in f.name for k in ctl_keywords)]

if not ctl_files:
    print("No new .ctl files found.")
    exit(1)

# Process ctl files
new_files_processed = False
updated_incomplete_ctl = {}
all_ctl_files = []

for ctl_file in ctl_files:
    ctl_file_name = os.path.basename(ctl_file.path)
    match = re.search(r'\.(\d{8})\.ctl$', ctl_file_name)
    timestamp = match.group(1) if match else "unknown"
    ctl_date = datetime.strptime(timestamp, "%Y%m%d") if timestamp != "unknown" else now

    if ctl_file_name in latest_processed:
        continue

    if ctl_file_name in incomplete_ctl:
        first_seen = datetime.strptime(incomplete_ctl[ctl_file_name]['first_seen'], "%Y-%m-%d")
        if now.date() > first_seen.date() + timedelta(days=2):
            print(f"Skipping {ctl_file_name}: too old and incomplete.")
            continue

    all_ctl_files.append((ctl_file, ctl_file_name, timestamp, ctl_date))

# Include previous incomplete ctl files (within 2-day window)
for name, record in incomplete_ctl.items():
    if name not in [c[1] for c in all_ctl_files]:
        if datetime.strptime(record['first_seen'], "%Y-%m-%d") + timedelta(days=2) >= now:
            path = os.path.join(weekly_base_path, f"dt={record['dt']}", name)
            all_ctl_files.append((dbutils.fs.ls(path)[0], name, record['timestamp'], datetime.strptime(record['timestamp'], "%Y%m%d")))

comparison_data_all = []

for ctl_file, ctl_file_name, timestamp, ctl_date in all_ctl_files:
    print(f"Evaluating: {ctl_file_name}")
    ctl_df = spark.read.text(ctl_file.path)
    ctl_data = ctl_df.collect()

    gz_list_from_ctl = []
    ctl_row_counts = {}

    for row in ctl_data:
        gz_filename, row_count = row[0].split('|')
        gz_list_from_ctl.append(gz_filename)
        ctl_row_counts[gz_filename] = int(row_count)

    gz_files = [f.path for f in files if f.name.endswith(".gz")]
    gz_files_with_info = []

    for gz_file in gz_files:
        gz_file_info = dbutils.fs.ls(gz_file)[0]
        s3_key_full = gz_file_info.path.replace("s3a://", "")
        s3_bucket, s3_key = s3_key_full.split("/", 1)
        s3_object = s3.head_object(Bucket=s3_bucket, Key=s3_key)
        last_modified = s3_object['LastModified']
        received_date = last_modified.strftime('%Y-%m-%d')
        gz_df = spark.read.csv(gz_file)
        gz_row_count = gz_df.count() - 1
        gz_files_with_info.append((gz_file_info.name, received_date, gz_row_count))

    comparison_data = []
    missing_gz_files = []

    for gz_name in gz_list_from_ctl:
        gz_info = next((g for g in gz_files_with_info if g[0] == gz_name), None)
        if gz_info:
            comparison_data.append((gz_name, gz_info[1], gz_info[2], ctl_row_counts[gz_name]))
        else:
            missing_gz_files.append(gz_name)

    if missing_gz_files:
        print(f"❌ Missing .gz for {ctl_file_name}: {missing_gz_files}")
        updated_incomplete_ctl[ctl_file_name] = {
            "timestamp": timestamp,
            "first_seen": incomplete_ctl.get(ctl_file_name, {"first_seen": now.strftime("%Y-%m-%d")})['first_seen'],
            "dt": latest_folder.split("=")[-1]
        }
        continue

    if comparison_data:
        new_files_processed = True
        comparison_data_all.extend(comparison_data)

        comparison_df = spark.createDataFrame(
            comparison_data,
            ["File_Name", "Date_Received_by_APM", "Row_Count_in_gz_File", "Row_Count_in_ctl_File"]
        )

        final_df = comparison_df \
            .withColumn("Status", when(col("Row_Count_in_ctl_File") == col("Row_Count_in_gz_File"), "SUCCESS").otherwise("MISMATCH")) \
            .withColumn("Number_of_Records_Received", col("Row_Count_in_ctl_File").cast("string")) \
            .withColumn("Number_of_Records_Loaded", col("Row_Count_in_gz_File").cast("string")) \
            .select(
                "File_Name",
                "Date_Received_by_APM",
                "Status",
                "Number_of_Records_Received",
                "Number_of_Records_Loaded"
            )

        final_df.write.format("delta").mode("append").saveAsTable("oh_apm_stg.vendor_extracts.APM_CPC_Data_Load_Report_Staging")
        print(f"✅ Loaded to table: {ctl_file_name}")

        latest_processed[ctl_file_name] = {
            "ctl_timestamp": timestamp,
            "processed_on": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        if final_df.filter(col("Status") == "MISMATCH").count() > 0:
            print("❌ Row count mismatch detected.")
            exit(1)

# Save state files
with open(latest_processed_path, 'w') as f:
    json.dump(latest_processed, f, indent=2)

with open(incomplete_ctl_path, 'w') as f:
    json.dump(updated_incomplete_ctl, f, indent=2)

# ✅ Set received_date if we processed valid files
if comparison_data_all:
    received_date = comparison_data_all[0][1]
    dbutils.jobs.taskValues.set(key="received_date", value=received_date, debugValue=received_date)
    print(f"✅ Set received_date: {received_date}")
else:
    print("⚠️ No new .ctl + .gz file matches found.")
    dbutils.jobs.taskValues.set(key="received_date", value=None)
